In [1]:
!pip install eyepop==3.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 50.0.0
    Uninstalling cryptography-50.0.0:
      Successfully uninstalled cryptography-50.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 26.4.0 requires cryptography<51,>=49.0.0, but you have cryptography 46.0.7 which is incompatible.


In [2]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

Enter your Account UUID: a5184defa8e847248f589d35080efbfa
Enter your API KEY: ··········


In [3]:
NAMESPACE_PREFIX="datasciencealliance-org" # Add your namespace-prefix here

### Define Ability

In [4]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import InferenceComponent, Pop
import json


ROAD_SURFACE_DAMAGE_DESCRIPTION_PROMPT = """
You are given an image of a road captured from a dash camera, vehicle-mounted camera, or municipal road survey camera.
Your task is to inspect the visible road surface and identify and describe all clearly visible road damage. For each damaged area, determine the damage type, visible severity, approximate location within the image, and a brief description.
Return ONLY valid JSON. Do not include explanations, markdown, or commentary outside the JSON object.

## ROAD CONDITION
Set "road_condition" to exactly one of: "smooth" — the visible road surface shows no clearly visible road damage, "damaged" — one or more clearly visible road defects are present.
If the road is smooth, set "overall_severity" to null and return an empty "damages" array.
Base the assessment only on clearly visible evidence. Do not report possible damage that cannot be identified confidently.

## DAMAGE TYPES
For each clearly visible damaged area, set "damage_type" to exactly one of: "pothole", "crack", "faded_road_marking", "other".

### "pothole"
A localized area where the road surface is visibly broken, recessed, or sunken relative to the surrounding road. Identify a pothole based on visible evidence of physical deformation or a damaged boundary, not darkness or color difference alone. Report clearly separate potholes as separate damage entries.

### "crack"
A visible linear fracture or break in the road surface. A crack may be straight, curved, branching, or irregular. Connected or branching cracks that form one continuous damaged region should be reported as one crack entry, while clearly separate crack regions should be reported separately.

### "faded_road_marking"
A painted road marking that shows visible loss or deterioration of paint. Report a marking when portions of its paint are noticeably faint, worn away, patchy, broken, or missing compared with other visible portions of the same marking or nearby road markings. A road marking is not faded when its paint appears consistently strong and clearly visible.

### "other"
Clearly visible physical road-surface damage that does not reasonably fit "pothole", "crack", or "faded_road_marking".

## INSPECTION RULES
Inspect the entire visible road surface before producing the response. Check for all supported damage types: pothole, crack, faded_road_marking, other.
Do not stop after finding the largest or most visually prominent defect. Report all clearly visible damaged regions. Group connected damage of the same type into one entry, and report clearly separate damaged regions as separate entries.
Report only damage clearly supported by visible evidence. If a possible defect cannot be confidently identified as one of the supported damage types, omit it from "damages". Do not guess or invent damage.

## WHAT IS NOT ROAD DAMAGE
Only report visible physical deterioration of the road surface or painted road markings.
Do not classify objects or visual effects as road damage, including vehicles, people, bicycles, traffic equipment, vegetation, buildings, shadows, reflections, debris, tire marks, normal road-surface texture, or normal road-surface color variation.

## DAMAGE LOCATION
For each damaged area, provide a brief description of where the damage appears within the WHOLE IMAGE.
Describe the approximate horizontal position as "left", "center", or "right", and the approximate vertical position as "upper", "middle", or "lower". Combine them into a short location description, such as "lower left of the image", "lower center of the image", "middle right of the image", or "upper center of the image".
For elongated or widespread damage such as cracks or faded road markings, describe the main area occupied by the damage or the direction in which it extends, such as "extends from the lower left toward the middle center of the image" or "extends across the middle of the image".
Always determine location relative to the WHOLE IMAGE, not relative to the road centerline, lanes, road edges, direction of travel, or vehicle position. Do not provide GPS coordinates, street names, addresses, or physical distances.

## SEVERITY
Set "severity" to exactly one of: "minor" — small or limited visible damage that is clearly present but limited in size or extent, "moderate" — clearly visible damage affecting a noticeable portion of the road surface that is significant but does not appear severe enough to be likely to cause an accident for vehicles traveling on the road, "severe" — substantial, extensive, or widespread visible damage that appears serious enough to create a significant driving hazard or contribute to an accident.
Determine severity only from what is visually apparent in the image, considering the visible size, extent, and condition of the damaged area. Do not estimate exact dimensions, structural integrity, repair costs, or required repairs.

## OVERALL SEVERITY
Determine "overall_severity" from both the severity of individual damaged areas and the amount and extent of damage across the visible road.
Only a small amount of minor damage → "minor", at least one moderate damaged area → normally at least "moderate", at least one severe damaged area → "severe", several minor damaged areas affecting a noticeable portion of the road → may be "moderate", several moderate damaged areas affecting a substantial portion of the road → may be "severe", no visible damage → null.
Do not determine "overall_severity" only from the single largest defect. Consider the combined visible condition of the road.

## DAMAGE DESCRIPTION
For each damaged area, provide a short factual description of its visible characteristics. Keep the description concise and describe only what can actually be seen.
Examples: "A localized depression is visible in the road surface.", "A branching crack extends across the road surface.", "The painted road marking is visibly worn and faded."
Do not speculate about the cause of the damage, recommend repairs, estimate measurements, or estimate repair costs.

## SUMMARY
Provide a brief overall description of the visible road condition in "summary". If damage is present, summarize all distinct damage types reported in "damages". If the road is smooth, state that the visible road surface appears smooth with no clear road damage. Do not describe unrelated objects or surroundings.

## REQUIRED JSON STRUCTURE
Return this exact JSON structure:
{
  "road_condition": null,
  "overall_severity": null,
  "summary": null,
  "damages": [
    {
      "damage_type": null,
      "severity": null,
      "location": null,
      "description": null
    }
  ]
}
Use only the allowed values defined above. Return only the JSON object.
"""


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.describe.road-surface-damage-description",
        description="Given an image of a road, describe the quality of the road. If potholes and damages are found describe the severity and location. If not the road would be described as smooth and free of damage.",
        worker_release="qwen3-instruct",
        text_prompt=ROAD_SURFACE_DAMAGE_DESCRIPTION_PROMPT,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=800,
            image_size=640
        ),
        is_public=False
    )
]

### Create Ability

In [5]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

created ability 06a91cf1b55178388000a2b73310d8f2 with alias entries [AbilityAliasEntry(alias='datasciencealliance-org.describe.road-surface-damage-description', tag='1.0.17'), AbilityAliasEntry(alias='datasciencealliance-org.describe.road-surface-damage-description', tag='latest')]


### Evalulate on a Single Image

In [6]:
from pathlib import Path
import json

pop = Pop(components=[
    InferenceComponent(
        ability=f"{NAMESPACE_PREFIX}.describe.road-surface-damage-description:latest"
    )
])

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
    endpoint.set_pop(pop)

    sample_img_path = Path("/content/sample_road_image.png")

    job = endpoint.upload(sample_img_path)
    result = job.predict()

print("=== RAW EYEPOP RESULT ===")
print(json.dumps(result, indent=2))

print("\n=== ROAD SURFACE DAMAGE DESCRIPTION OUTPUT ===")
texts = result.get("texts", [])

if not texts:
    print("No text output found. Check that the image path is correct and the ability alias was published successfully.")
else:
    for text_item in texts:
        raw_text = text_item.get("text", "")

        try:
            parsed = json.loads(raw_text)
            print(json.dumps(parsed, indent=2))
        except json.JSONDecodeError:
            print(raw_text)

=== RAW EYEPOP RESULT ===
{
  "compute_units": 0.609,
  "seconds": 0,
  "source_height": 1086,
  "source_id": "0cbedb2a-a30c-11f1-82d5-9aa95ef2f031",
  "source_width": 1448,
  "system_timestamp": 1787940774153731000,
  "texts": [
    {
      "id": 1,
      "text": "{\n  \"road_condition\": \"damaged\",\n  \"overall_severity\": \"moderate\",\n  \"summary\": \"The visible road surface shows multiple distinct potholes and extensive cracking, indicating moderate damage.\",\n  \"damages\": [\n    {\n      \"damage_type\": \"pothole\",\n      \"severity\": \"moderate\",\n      \"location\": \"lower left of the image\",\n      \"description\": \"A large, localized depression with a broken, sunken surface is visible in the left lane.\"\n    },\n    {\n      \"damage_type\": \"pothole\",\n      \"severity\": \"moderate\",\n      \"location\": \"lower right of the image\",\n      \"description\": \"A localized depression with a broken, sunken surface is visible in the right lane.\"\n    },\n    